# Auralis merge: average the three branches, score every candidate, export the best

Attached as kernel inputs: the LibriSpeech branch (`auralis-train-stage2`), the People's Speech branch (`auralis-train-peoples`) and the VoxPopuli branch (`auralis-train-vox`). Each was fine-tuned from the same stage 1 checkpoint, so averaging their weights is meaningful.

The notebook scores every branch and the merged model on test-clean and test-other, then exports whichever has the lowest mean WER. A merge is not guaranteed to beat the best branch, so the table is the deciding evidence.

In [ ]:
import glob, json, os, shutil, subprocess, sys

SOURCES = {'librispeech': 'auralis-train-stage2', 'peoples': 'auralis-train-peoples', 'voxpopuli': 'auralis-train-vox'}
TEST_SETS = ['test-clean', 'test-other']
SUBSETS = []
TEMP = '/kaggle/temp'
LS = f'{TEMP}/librispeech'
WORK = f'{TEMP}/work'
for d in (LS, WORK, '/kaggle/working'):
    os.makedirs(d, exist_ok=True)

In [ ]:
!cd /kaggle/temp && rm -rf Auralis && git clone --depth 1 --branch feat/train-recipe https://github.com/Sagexd08/Auralis.git
!nvidia-smi -L
!pip install -q torch --index-url https://download.pytorch.org/whl/cu126
!pip install -q soundfile scipy onnx onnxscript onnxruntime jiwer
import sys
if 'torch' in sys.modules:
    raise RuntimeError('torch was already imported in this session. Run > Restart session, then run all cells again.')
import torch
print(torch.__version__, 'cuda build:', torch.version.cuda)
assert torch.cuda.is_available(), 'torch cannot see a GPU. If nvidia-smi -L above printed no GPU, set Accelerator to GPU T4 x2 in the notebook settings.'
print(torch.cuda.get_device_name(0))

In [ ]:
import tarfile
import hashlib
import urllib.request

REGISTRY = f'{WORK}/registry.jsonl'
DATA_CLI = ['python', '-m', 'auralis_data', '--registry', REGISTRY]
DATA_DIR = '/kaggle/temp/Auralis/data'

def run(cmd, cwd):
    subprocess.run(cmd, cwd=cwd, check=True)

def stream_extract(url, dest):
    with urllib.request.urlopen(url) as response:
        with tarfile.open(fileobj=response, mode='r|gz') as tar:
            tar.extractall(dest, filter='data')

for name in TEST_SETS:
    root = f'{LS}/LibriSpeech/{name}'
    if not os.path.isdir(root):
        stream_extract(f'https://www.openslr.org/resources/12/{name}.tar.gz', LS)
    run(DATA_CLI + ['import', 'librispeech', root, '--id', f'librispeech-{name}', '--language', 'en', '--revision', name, '--manifests', f'{WORK}/manifests'], DATA_DIR)
    cfg = {'name': f'librispeech-{name}', 'datasets': [f'librispeech-{name}'], 'commercial': True, 'min_duration': 1.0, 'max_duration': 35.0, 'min_quality': 0.8, 'seed': 1004}
    path = f'{WORK}/{name}.config.json'
    with open(path, 'w') as f:
        json.dump(cfg, f)
    run(DATA_CLI + ['build', path, '--manifests', f'{WORK}/manifests', '--out', f'{WORK}/corpora'], DATA_DIR)
    print('ready', name)

In [ ]:
sys.path.insert(0, '/kaggle/temp/Auralis/training')
from auralis_stt.merge import average_checkpoints

def newest(paths):
    paths = [p for p in paths if os.path.isdir(p)]
    return max(paths, key=lambda p: int(p.rsplit('-', 1)[1])) if paths else None

candidates = {}
for label, slug in SOURCES.items():
    found = newest(glob.glob(f'/kaggle/input/{slug}/**/auralis-stt-step-*', recursive=True))
    print(label, '->', found)
    if found:
        candidates[label] = found

assert len(candidates) >= 2, 'need at least two branch checkpoints attached as inputs'
merged_dir = average_checkpoints(list(candidates.values()), '/kaggle/working/merged/auralis-stt-step-0')
candidates['merged'] = str(merged_dir)
print('merged', len(candidates) - 1, 'branches')

In [ ]:
import torch
from torch.utils.data import DataLoader
from auralis_stt.checkpoint import load_model
from auralis_stt.dataset import SpeechDataset, collate, load_corpus
from auralis_stt.features import LogMel
from auralis_stt.train import evaluate

device = torch.device('cuda')
table = {}
for label, ckpt in candidates.items():
    model, tokenizer = load_model(ckpt, 'cuda')
    logmel = LogMel(n_mels=model.cfg.n_mels).to('cuda')
    table[label] = {}
    for t in TEST_SETS:
        meta, rows, roots = load_corpus(f'{WORK}/corpora', f'librispeech-{t}')
        loader = DataLoader(SpeechDataset(rows, roots, tokenizer, 35.0), 16, shuffle=False, collate_fn=collate, num_workers=2)
        metrics, _, _ = evaluate(model, tokenizer, loader, logmel, device)
        table[label][t] = metrics
    mean_wer = sum(table[label][t]['wer'] for t in TEST_SETS) / len(TEST_SETS)
    table[label]['mean_wer'] = mean_wer
    print(label, {t: round(table[label][t]['wer'] * 100, 2) for t in TEST_SETS}, 'mean', round(mean_wer * 100, 2))

best = min(table, key=lambda k: table[k]['mean_wer'])
print('best:', best)
with open('/kaggle/working/merge_results.json', 'w') as f:
    json.dump({'decoding': 'greedy CTC', 'checkpoints': candidates, 'results': table, 'best': best}, f, indent=2)

In [ ]:
from auralis_stt.export import export_checkpoint
EXPORT_DIR = '/kaggle/working/export/auralis-v1'
onnx_path = export_checkpoint(candidates[best], EXPORT_DIR, 'auralis', ['en'])
shutil.copy('/kaggle/working/merge_results.json', EXPORT_DIR)
shutil.make_archive('/kaggle/working/auralis-v1', 'zip', EXPORT_DIR)
print(onnx_path)